In [146]:
#Importação de Bibliotecas

import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

In [147]:
#Exportando dataset limpo
df = pd.read_csv("/content/sample_data/udemy_cursos_completos.csv")

df.head()

,course_id,course_title,url,price,num_subscribers,num_reviews,num_lectures,level,Rating,content_duration,published_timestamp,subject
0,49798.0,bitcoin or how i learned to stop worrying and ...,https://www.udemy.com/bitcoin-or-how-i-learned...,0.0,65576.0,936.0,24.0,All Levels,0.56,8.0,2013-04-20T02:25:22Z,Business Finance
1,48841.0,accounting in 60 minutes a brief introduction,https://www.udemy.com/accounting-in-60-minutes...,0.0,56659.0,4397.0,16.0,Beginner Level,0.95,1.5,2013-04-07T21:39:25Z,Business Finance
2,133536.0,stock market investing for beginners,https://www.udemy.com/the-beginners-guide-to-t...,0.0,50855.0,2698.0,15.0,All Levels,0.91,1.5,2013-12-25T19:53:34Z,Business Finance
3,151668.0,introduction to financial modeling,https://www.udemy.com/financial-modeling-asimp...,0.0,29167.0,1463.0,8.0,All Levels,0.18,1.5,2014-05-27T16:22:16Z,Business Finance
4,648826.0,the complete financial analyst course 2017,https://www.udemy.com/the-complete-financial-a...,195.0,24481.0,2347.0,174.0,All Levels,0.37,10.0,2016-01-21T01:38:48Z,Business Finance


In [148]:
# Remover cursos que não tiveram relevância

df = df[df['num_subscribers'] > 0]


In [149]:
# Isso melhora a relevância das recomendações
# Duplicando título
df['conteudo'] = (
    df['course_title'].str.lower() + " " +
    df['course_title'].str.lower() + " " +
    df['subject'].str.lower() + " " +
    df['level'].str.lower()
)

In [150]:
#remoção de algumas palavras genéricas
extra_stop = {
    "learn", "learned", "learning", "how", "you", "your",
    "course", "complete", "guide", "part", "beginner", "level"
}

stop_words = list(ENGLISH_STOP_WORDS.union(extra_stop))

# Remove títulos muito curtos
df = df[df['course_title'].str.split().str.len() > 3]

In [151]:
# Remove cursos com títulos em outros idiomas (mantém apenas inglês)
def is_english(text):
    return bool(re.match(r'^[\x00-\x7F]+$', text))

df = df[df['course_title'].apply(is_english)]

In [152]:
# Converte o texto em números (matriz)
vectorizer = TfidfVectorizer(
    stop_words=stop_words,
    max_df=0.8,
    min_df=1,
    ngram_range=(1,2),
    max_features=5000
)

tfidf_matrix = vectorizer.fit_transform(df['conteudo'])

In [153]:
# Mede as similaridades dos cursos
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [154]:
# Padroniza títulos para busca
df['course_title_clean'] = df['course_title'].str.lower()

# Cria um índice: nome do curso e posição no dataset
indices = pd.Series(df.index, index=df['course_title_clean']).drop_duplicates()

In [155]:
# Função de recomendação

def recomendar(nome_curso, top_n=5):
    nome_curso = nome_curso.lower()

# Verifica se o curso existe
    if nome_curso not in indices:
        return "Curso não encontrado"

    idx = indices[nome_curso]

# Ordena do mais similar para o menos similar
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    top = scores[1:top_n+10]

# Seleciona os cursos recomendados
    resultados = df.iloc[[i[0] for i in top]].copy()
# Filtro por categoria
    subject_base = df.iloc[idx]['subject']

    resultados = resultados.drop_duplicates(subset='course_title')

 # Ordena por popularidade (mais inscritos primeiro)
    resultados = resultados.sort_values(by='num_subscribers', ascending=False)
    return resultados.head(top_n)[
        ['course_title', 'subject', 'level', 'num_subscribers']
    ]

In [156]:
curso = df['course_title'].iloc[0]
print("Curso base:", curso)

recomendar(curso)

Curso base: bitcoin or how i learned to stop worrying and love crypto


,course_title,subject,level,num_subscribers
20,bitcoin for beginners your quick start guide t...,Business Finance,All Levels,10670.0
34,the complete bitcoin course get 001 bitcoin in...,Business Finance,All Levels,8797.0
204,bitcoin the complete guide,Business Finance,All Levels,2281.0
576,how to confidently join the bitcoin revolution,Business Finance,All Levels,547.0
582,learn about bitcoin and bitcoin mining,Business Finance,All Levels,520.0


In [157]:
curso = df['course_title'].iloc[1100]
print("Curso base:", curso)

recomendar(curso)

Curso base: vector logo design in affinity designer


,course_title,subject,level,num_subscribers
1204,photoshop in ease create world amazing graphi...,Graphic Design,Beginner Level,14440.0
1211,create professional looking infographics with ...,Graphic Design,All Levels,9367.0
2824,create a professional website without programm...,Subject: Web Development,Beginner Level,5421.0
2845,wordpress for beginners create a professional ...,Subject: Web Development,Beginner Level,5129.0
1284,adobe create a professional logo stepbystep b...,Graphic Design,All Levels,2420.0


In [158]:
curso = df['course_title'].iloc[2500]
print("Curso base:", curso)

recomendar(curso)

Curso base: learn ruby programming the easy way


,course_title,subject,level,num_subscribers
2475,the web developer bootcamp,Subject: Web Development,Beginner Level,121584.0
2477,the complete web developer course 20,Subject: Web Development,Beginner Level,114512.0
2486,become a web developer from scratch,Subject: Web Development,All Levels,69186.0
2510,the original complete web developer course,Subject: Web Development,Beginner Level,31070.0
2526,the ultimate web developer how to guide,Subject: Web Development,Beginner Level,24861.0
